# CV Masterclass 14: Super-Resolution & Image Restoration

Welcome to Notebook 16. In previous modules, we focused on **Extracting Information** from images (Detection, Classification, OCR). Today, we focus on **Synthesizing Information** that was never there. 

Super-Resolution (SR) and Image Restoration are the "CSI-Enhance" of computer vision. We take a degraded, blurry, or low-resolution image and attempt to reconstruct the underlying high-resolution reality. This is not just a resizing problem; it is an **Inverse Problem** that requires the model to "hallucinate" plausible high-frequency textures (hair, skin pores, fabric) based on learned priors.

---

## 🎓 Socratic Deep Check
Ponder these questions before we begin:

> *"If a $32 \times 32$ image has 1,024 pixels, and we want to upsample it to $128 \times 128$ (16,384 pixels), where do the missing 15,360 values come from? If there is no information in the source, can a model truly 'know' the answer, or is it merely guessing?"*

> *"Mathematical fidelity (low MSE) and Perceptual quality (looking 'real') are often at odds. If a model generates a perfectly sharp eye that doesn't actually match the original person's eye shape, is the model 'accurate' or has it 'failed'?"*

---

## Table of Contents
1. **The Ill-Posed Math:** The Degradation Model and Null Space.
2. **The Perception-Distortion Tradeoff:** Why MSE makes things blurry.
3. **Architectural Evolution:** SRCNN, ESRGAN (RRDB), and Real-World Degradation.
4. **Transformer-based Restoration:** SwinIR and Non-local Self-Attention.
5. **Loss Functions:** Pixel Loss vs. Perceptual Loss (LPIPS) vs. Adversarial Loss.
6. **Implementation:** Building Perceptual Loss in PyTorch.
7. **Common Pitfalls:** Artifacts and Training Instability.

## 1. The Ill-Posed Math: The Degradation Model

To understand Super-Resolution, we must first understand how an image gets ruined. The **Degradation Model** describes the relationship between the High-Resolution (HR) image $x$ and the Low-Resolution (LR) image $y$:

$$ y = (x \otimes k) \downarrow_s + \mathbf{n} $$

Where:
- $x$: The original High-Resolution ground truth.
- $k$: A blur kernel (PSF - Point Spread Function).
- $\otimes$: Convolution operator.
- $\downarrow_s$: Downsampling operator with scale factor $s$.
- $\mathbf{n}$: Additive White Gaussian Noise (AWGN).

### The "Ill-Posed" Nature
Super-Resolution is the attempt to solve for $x$ given $y$. This is **Ill-Posed** because for every LR image $y$, there are an **infinite number** of possible HR images $x$ that could have produced it (this is the **Null Space** of the degradation operator). 

Because $H$ (the degradation) is a many-to-one mapping, its inverse $H^{-1}$ does not exist. We must use **statistical priors** (Deep Learning) to select the most "plausible" $x$ from the set of possible solutions.

## 2. The Perception-Distortion Tradeoff

In traditional ML, we minimize **Mean Squared Error (MSE)**. However, in Super-Resolution, minimizing MSE leads to a specific behavior: the model predicts the **average** of all possible high-resolution realities.

### The MSE Trap
Imagine the LR pixel corresponds to an area that could be either a "sharp edge left" or a "sharp edge right". The MSE-optimal solution is the average of both: a **blurry edge**. This is why SR models trained only on $L_2$ loss look soft and plastic-like.

### Blau & Michal's Theorem (2018)
There is a fundamental tradeoff between **Distortion** (mathematical fidelity, e.g., PSNR/MSE) and **Perceptual Quality** (how "real" it looks to a human, e.g., LPIPS/NIQE).

- As you decrease distortion, you often increase unnatural artifacts.
- As you increase perceptual quality (making things look sharp), you often deviate mathematically from the ground truth (MSE goes up).

| Objective | Loss Function | Visual Result |
|---|---|---|
| **Fidelity** | Pixel Loss ($L_1$/$L_2$) | High PSNR, but blurry/soft. |
| **Perception** | Perceptual + Adversarial | Sharp textures, but lower PSNR. |

## 3. Architectural Evolution: SRCNN to ESRGAN

### SRCNN (2014)
The first deep learning SR model used only 3 convolutional layers. It proved that deep learning could outperform bicubic interpolation but suffered from extreme blurriness.

### EDSR (Enhanced Deep Super-Resolution)
EDSR removed **Batch Normalization (BN)**. Why? Because BN normalizes the features based on the batch statistics, which destroys the absolute pixel values and range information critical for reconstructing fine-scale textures.

### ESRGAN & RRDB (2018)
ESRGAN introduced the **Residual-in-Residual Dense Block (RRDB)**. Instead of simple residuals, it uses a "dense-in-residual" structure that allows the gradients to flow through a massive number of skip connections, enabling the learning of extremely complex high-frequency features.

In [ ]:
import torch
import torch.nn as nn

class ResidualDenseBlock(nn.Module):
    """
    The core building block of ESRGAN.
    Combines residual connections with dense connections for max feature reuse.
    """
    def __init__(self, nf=64, gc=32):
        super().__init__()
        # nf: number of features, gc: growth channel
        self.conv1 = nn.Conv2d(nf, gc, 3, 1, 1)
        self.conv2 = nn.Conv2d(nf + gc, gc, 3, 1, 1)
        self.conv3 = nn.Conv2d(nf + 2 * gc, gc, 3, 1, 1)
        self.conv4 = nn.Conv2d(nf + 3 * gc, gc, 3, 1, 1)
        self.conv5 = nn.Conv2d(nf + 4 * gc, nf, 3, 1, 1)
        self.lrelu = nn.LeakyReLU(negative_slope=0.2, inplace=True)

    def forward(self, x):
        x1 = self.lrelu(self.conv1(x))
        x2 = self.lrelu(self.conv2(torch.cat((x, x1), 1)))
        x3 = self.lrelu(self.conv3(torch.cat((x, x1, x2), 1)))
        x4 = self.lrelu(self.conv4(torch.cat((x, x1, x2, x3), 1)))
        x5 = self.conv5(torch.cat((x, x1, x2, x3, x4), 1))
        # Residual scaling (standard technique to stabilize deep SR training)
        return x5 * 0.2 + x

class RRDB(nn.Module):
    """ Residual in Residual Dense Block """
    def __init__(self, nf=64, gc=32):
        super().__init__()
        self.RDB1 = ResidualDenseBlock(nf, gc)
        self.RDB2 = ResidualDenseBlock(nf, gc)
        self.RDB3 = ResidualDenseBlock(nf, gc)

    def forward(self, x):
        out = self.RDB1(x)
        out = self.RDB2(out)
        out = self.RDB3(out)
        return out * 0.2 + x

# TEST IT
model = RRDB(nf=64, gc=32)
dummy_input = torch.randn(1, 64, 32, 32)
output = model(dummy_input)
print(f"Input shape: {dummy_input.shape}")
print(f"Output shape: {output.shape}")

assert output.shape == dummy_input.shape
print("RRDB Implementation Verified!")

## 3.1. Real-World Degradation: The Key Gap Between Academic and Production SR

Models trained on standard bicubic downsampling fail on real photos because real degradation is **NOT** bicubic. Real images suffer from lens blur, sensor noise, JPEG compression, film grain, chromatic aberration, and motion blur — all applied in random combinations.

**Real-ESRGAN** introduced a **"High-Order Degradation Pipeline"**: instead of one degradation step, apply 2 sequential random degradation rounds:
1.  **Round 1:** Random blur → noise → JPEG compression → resize
2.  **Round 2:** Another random blur → noise → JPEG → resize

This mimics the kind of compounding degradation seen in:
- **Phone photos posted to Facebook:** (resize → JPEG → re-download → JPEG)
- **Old VHS footage:** (analog noise → digitization artifacts → compression)
- **Scanned documents:** (scan noise → JPEG storage → print → scan again)


In [ ]:
import numpy as np
import torch
import cv2

def simulate_real_world_degradation(img_tensor, severity='medium'):
    """
    Simulates Real-ESRGAN-style two-round complex degradation.
    img_tensor: (C, H, W) float32 in [0, 1]
    Returns: degraded image tensor
    """
    severity_params = {
        'light':  {'blur_range': (1, 5),  'noise_std': (0, 10),  'jpeg_q': (70, 95)},
        'medium': {'blur_range': (3, 15), 'noise_std': (5, 30),  'jpeg_q': (40, 80)},
        'heavy':  {'blur_range': (7, 25), 'noise_std': (20, 60), 'jpeg_q': (10, 50)},
    }
    p = severity_params[severity]
    
    # Convert to numpy (H, W, C), uint8
    img_np = (img_tensor.permute(1, 2, 0).numpy() * 255).astype(np.uint8)
    
    def apply_round(img):
        # 1. Random Gaussian blur
        ksize = np.random.choice(range(p['blur_range'][0], p['blur_range'][1]+1, 2))
        img = cv2.GaussianBlur(img, (ksize, ksize), 0)
        
        # 2. Random additive noise
        std = np.random.uniform(*p['noise_std'])
        noise = np.random.normal(0, std, img.shape).astype(np.float32)
        img = np.clip(img.astype(np.float32) + noise, 0, 255).astype(np.uint8)
        
        # 3. JPEG compression
        quality = np.random.randint(*p['jpeg_q'])
        _, enc = cv2.imencode('.jpg', img, [cv2.IMWRITE_JPEG_QUALITY, quality])
        img = cv2.imdecode(enc, cv2.IMREAD_COLOR)
        
        return img
    
    # Two rounds of degradation
    degraded = apply_round(img_np)
    degraded = apply_round(degraded)
    
    # Back to tensor
    result = torch.from_numpy(degraded.astype(np.float32) / 255.0)
    return result.permute(2, 0, 1)

# TEST IT
clean_img = torch.rand(3, 64, 64)

for severity in ['light', 'medium', 'heavy']:
    degraded = simulate_real_world_degradation(clean_img, severity)
    assert degraded.shape == clean_img.shape
    psnr = -10 * torch.log10(((clean_img - degraded)**2).mean())
    print(f"{severity:8s} degradation PSNR: {psnr:.2f} dB")

# PSNR should decrease as severity increases
light = simulate_real_world_degradation(clean_img, 'light')
heavy = simulate_real_world_degradation(clean_img, 'heavy')
psnr_light = -10 * torch.log10(((clean_img - light)**2).mean())
psnr_heavy = -10 * torch.log10(((clean_img - heavy)**2).mean())
assert psnr_light > psnr_heavy, "Heavier degradation must lower PSNR!"
print("Real-world degradation pipeline verified ✓")

## 4. SwinIR: Transformer-based Restoration

The current state-of-the-art in restoration has shifted from CNNs to transformers. **SwinIR** uses the Swin Transformer architecture to solve image restoration.

**Why Transformers for SR?**
- **Non-local dependencies:** High-resolution restoration often needs to look at similar patches far away (e.g., repeating patterns on a building).
- **Shifted Windows:** By computing self-attention within windows and then shifting those windows, the model maintains a massive receptive field with linear complexity relative to image size.
- **Weighted Feature Aggregation:** Unlike convolutions (fixed weights), attention dynamically focuses on relevant content for restoration.

In [ ]:
class SwinIRBlock(nn.Module):
    """
    Simplified Residual Swin Transformer Block for image restoration.
    Key difference from standard Swin: residual connection wraps the ENTIRE
    block, allowing shallow features to bypass deep transformations.
    This preserves low-frequency information (colors, global structure)
    while the Swin layers learn high-frequency corrections.
    """
    def __init__(self, dim=60, window_size=8, num_heads=6):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(dim, num_heads, batch_first=True)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(
            nn.Linear(dim, dim * 4),
            nn.GELU(),
            nn.Linear(dim * 4, dim),
        )
        # Conv projection for the residual (ensures channels match)
        self.residual_proj = nn.Conv2d(dim, dim, 3, padding=1)
        
    def forward(self, x):
        # x: (B, C, H, W)
        B, C, H, W = x.shape
        residual = x
        
        # Flatten spatial dimensions for attention
        x_flat = x.flatten(2).transpose(1, 2)  # (B, H*W, C)
        
        # Self-Attention
        x_norm = self.norm1(x_flat)
        attn_out, _ = self.attn(x_norm, x_norm, x_norm)
        x_flat = x_flat + attn_out
        
        # Feed-Forward
        x_norm2 = self.norm2(x_flat)
        x_flat = x_flat + self.mlp(x_norm2)
        
        # Reshape back
        x = x_flat.transpose(1, 2).view(B, C, H, W)
        
        # Residual connection (through a conv for local feature refinement)
        return residual + self.residual_proj(x)

# TEST IT
sr_block = SwinIRBlock(dim=60)
lr_features = torch.randn(2, 60, 32, 32)
sr_features = sr_block(lr_features)
assert sr_features.shape == lr_features.shape
print(f"SwinIR Block: {lr_features.shape} → {sr_features.shape}")
params = sum(p.numel() for p in sr_block.parameters())
print(f"Block Parameters: {params:,}")

## 5. Loss Functions: Beyond Pixel Loss

To train a high-quality restorer, we typically use a weighted sum of three losses:

1.  **Pixel Loss ($L_1$):** $\|x - \hat{x}\|_1$. Stabilizes training and ensures overall color/structure fidelity.
2.  **Perceptual Loss ($L_{feat}$):** Compares feature maps from a pre-trained network (usually VGG-19). It penalizes differences in "concepts" rather than pixels.
3.  **Adversarial Loss ($L_{adv}$):** Using a Discriminator to push the Generator to produce textures that "look real" (GAN training).

### Math of Perceptual Loss
$$ L_{percept} = \sum_{j} \frac{1}{C_j H_j W_j} \| \phi_j(x) - \phi_j(\hat{x}) \|_2^2 $$
Where $\phi_j$ is the $j$-th activation layer of a pre-trained VGG-19 network.

In [ ]:
from torchvision import models

class PerceptualLoss(nn.Module):
    """
    Feature-based loss using VGG-19.
    Standard in ESRGAN and StyleGAN.
    """
    def __init__(self):
        super().__init__()
        # Use Weights for modern torchvision API
        vgg = models.vgg19(weights='DEFAULT').features
        
        # We use levels relu1_2, relu2_2, relu3_4, relu4_4
        self.slice1 = nn.Sequential(*list(vgg.children())[:4])   # relu1_2
        self.slice2 = nn.Sequential(*list(vgg.children())[4:9])  # relu2_2
        self.slice3 = nn.Sequential(*list(vgg.children())[9:18]) # relu3_4
        self.slice4 = nn.Sequential(*list(vgg.children())[18:27])# relu4_4
        
        # VGG expects specifically normalized input
        self.register_buffer('mean', torch.Tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer('std', torch.Tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

        for param in self.parameters():
            param.requires_grad = False # Freeze VGG

    def forward(self, input, target):
        # Normalize according to ImageNet stats
        input = (input - self.mean) / self.std
        target = (target - self.mean) / self.std

        # Forward pass through slices
        h1 = self.slice1(input)
        h1_t = self.slice1(target)
        loss = torch.norm(h1 - h1_t, 2)
        
        h2 = self.slice2(h1)
        h2_t = self.slice2(h1_t)
        loss += torch.norm(h2 - h2_t, 2)

        h3 = self.slice3(h2)
        h3_t = self.slice3(h2_t)
        loss += torch.norm(h3 - h3_t, 2)

        h4 = self.slice4(h3)
        h4_t = self.slice4(h3_t)
        loss += torch.norm(h4 - h4_t, 2)
        
        return loss

# TEST IT
loss_fn = PerceptualLoss()
img1 = torch.randn(1, 3, 64, 64)
img2 = img1 + 0.01 * torch.randn(1, 3, 64, 64) # Slightly noisy
img3 = torch.randn(1, 3, 64, 64)               # Completely different

loss_similar = loss_fn(img1, img2)
loss_diff = loss_fn(img1, img3)

print(f"Loss (Similar): {loss_similar.item():.4f}")
print(f"Loss (Different): {loss_diff.item():.4f}")

assert loss_similar < loss_diff
print("Perceptual Loss behaves correctly!")

### COMMON PITFALLS
*   **Checkerboard Artifacts:** Often caused by Deconvolution (Transposed Convolution) layers. Use **Nearest Neighbor Upsampling + Convolution** instead to prevent grid patterns.
*   **Training Instability:** GAN-based SR is notoriously unstable. Start with a "Pre-training" phase using only $L_1$ loss (this creates a stable PSNR model) before introducing Perceptual/Adversarial losses.
*   **The Boundary Problem:** Padding in deep networks creates artifacts at the edges of the image. Always crop enough 'context' during training to avoid edge bias.

### 🎓 DEEP QUESTION ANSWERED

**Q:** *Where do the missing pixels come from if they aren't in the source?*

**A:** They come from the **learned manifold** of the training data. The model isn't "inventing" from thin air; it is performing a conditional lookup. It recognizes a set of LR pixels as "Likely Grass" and replaces them with a sharp texture of grass it saw during training. This is why SR models perform poorly on out-of-distribution data (e.g., an SR model trained only on faces will try to put eyes and lips on a picture of a car).

**Q:** *If a model creates a sharp eye that doesn't match the original, is it a failure?*

**A:** This depends on the application. For **Entertainment/Photography** (making old photos look good), it's a success. For **Medical Imaging/Forensics**, it's a catastrophic failure because it's "Hallucinating" evidence. This is the core of the Perception-Distortion tradeoff: do you want a sharp lie or a blurry truth?